In [ ]:
# Re-import after state reset
import random
import pandas as pd
from datetime import datetime, timedelta
import uuid

# Config
HOME_LOC = (28.6139, 77.2090)  # Delhi
CITY_POOL = {
    "Paris": (48.8566, 2.3522),
    "New York": (40.7128, -74.0060),
    "Tokyo": (35.6895, 139.6917),
    "Bangalore": (12.9716, 77.5946),
    "Goa": (15.2993, 74.1240),
}
SCENE_POOL = ["monument", "food", "beach", "street", "family", "selfie"]
FACES_POOL = [{"gender": g} for g in ["male", "female"]]
ACTIVITIES = ["payingAct", "eatingAct", "movingAct", "watchMap", "surfingNet", "watchVideo", "music", "shopping"]

# Output storage
taking_pic = []
staying_act = []
travel_coll = []
extra_activities = []

# Helper functions
def random_time_span(start_date, min_minutes=5, max_minutes=30):
    start_time = start_date + timedelta(hours=random.randint(8, 18), minutes=random.randint(0, 59))
    duration = timedelta(minutes=random.randint(min_minutes, max_minutes))
    return start_time, start_time + duration

def random_face_data(mode):
    count = random.randint(1, 2) if mode == "solo" else random.randint(2, 5)
    return [dict(id=str(uuid.uuid4())[:6], gender=random.choice(["male", "female"]), bbox=str((10,10,50,50))) for _ in range(count)]

# Generate data
start_date = datetime(2023, 1, 1)
pic_id_counter = 0
stay_id_counter = 0
coll_id_counter = 0
extra_act_counter = 0

for week in range(52):
    week_start = start_date + timedelta(days=week*7)

    travel_type = random.choice(["none", "visit", "short", "long"])
    travel_mode = random.choice(["solo", "family"])

    if travel_type == "none":
        continue

    location = random.choice(list(CITY_POOL.keys()))
    loc_latlon = CITY_POOL[location]

    travel_days = 1 if travel_type == "visit" else (2 if travel_type == "short" else random.randint(3, 6))
    travel_start = week_start + timedelta(days=random.randint(0, 3))
    travel_end = travel_start + timedelta(days=travel_days)

    stay_start = travel_start + timedelta(hours=2)
    stay_end = travel_end - timedelta(hours=2)
    stay_id = f"stay_{stay_id_counter}"
    staying_act.append({
        "id": stay_id,
        "start_time": stay_start.isoformat(),
        "end_time": stay_end.isoformat(),
        "location": location,
        "lat": loc_latlon[0],
        "lon": loc_latlon[1],
        "reason": "hotel stay"
    })
    stay_id_counter += 1

    # Taking pictures
    pic_ids = []
    for _ in range(random.randint(2, 4 if travel_mode == "solo" else 6)):
        span_start, span_end = random_time_span(travel_start)
        image_ids = [f"img_{uuid.uuid4().hex[:5]}" for _ in range(random.randint(2, 5))]
        scene_tags = random.sample(SCENE_POOL, random.randint(1, 3))
        faces = random_face_data(travel_mode)

        pic_id = f"pic_{pic_id_counter}"
        taking_pic.append({
            "id": pic_id,
            "start_time": span_start.isoformat(),
            "end_time": span_end.isoformat(),
            "images": ",".join(image_ids),
            "location": location,
            "lat": loc_latlon[0],
            "lon": loc_latlon[1],
            "scenes": ",".join(scene_tags),
            "faces": str(faces)
        })
        pic_ids.append(pic_id)
        pic_id_counter += 1

    # Extra activities
    for _ in range(random.randint(4, 8)):
        act_type = random.choice(ACTIVITIES)
        act_start, act_end = random_time_span(travel_start)
        extra_activities.append({
            "id": f"act_{extra_act_counter}",
            "type": act_type,
            "start_time": act_start.isoformat(),
            "end_time": act_end.isoformat(),
            "location": location,
            "lat": loc_latlon[0],
            "lon": loc_latlon[1]
        })
        extra_act_counter += 1

    # Travel collection entry
    coll_id = f"travel_{coll_id_counter}"
    travel_coll.append({
        "id": coll_id,
        "start_time": travel_start.isoformat(),
        "end_time": travel_end.isoformat(),
        "type": travel_type,
        "mode": travel_mode,
        "location": location,
        "pic_ids": ",".join(pic_ids),
        "stay_ids": stay_id
    })
    coll_id_counter += 1



In [ ]:
# Create DataFrames
df_pic = pd.DataFrame(taking_pic)
df_stay = pd.DataFrame(staying_act)
df_travel = pd.DataFrame(travel_coll)
df_extra = pd.DataFrame(extra_activities)


In [ ]:
df_pic.shape

(140, 9)

In [ ]:
df_travel.shape

(39, 8)

In [ ]:
df_extra.shape

(251, 7)

In [ ]:
# Re-run necessary imports and code after reset
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Load the generated CSV files again
# df_pic = pd.read_csv("/mnt/data/generated_df_pic.csv", parse_dates=["start_time", "end_time"])
# df_extra = pd.read_csv("/mnt/data/generated_df_extra.csv", parse_dates=["start_time", "end_time"])
# df_stay = pd.read_csv("/mnt/data/generated_df_stay.csv", parse_dates=["start_time", "end_time"])

# Tag activity types
df_pic["type"] = "picTaking"
df_extra["type"] = df_extra["type"].fillna("unknown")

# Combine for clustering
df_all = pd.concat([df_pic, df_extra], ignore_index=True)

# Convert timestamp for clustering
df_all["timestamp"] = df_all["start_time"].astype(np.int64) // 10**9
df_all["lat"] = df_all["lat"].astype(float)
df_all["lon"] = df_all["lon"].astype(float)

# Scale for DBSCAN
features = df_all[["timestamp", "lat", "lon"]]
X_scaled = StandardScaler().fit_transform(features)

# Apply DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=3)
df_all["cluster"] = dbscan.fit_predict(X_scaled)

# Summarize clusters
cluster_summary = df_all.groupby("cluster").agg(
    start_time=("start_time", "min"),
    end_time=("end_time", "max"),
    duration_sec=("timestamp", lambda x: x.max() - x.min()),
    activity_count=("type", "count"),
    unique_types=("type", pd.Series.nunique)
).reset_index()

# Classify moments
def classify_moment(row):
    if row["duration_sec"] > 3600 and row["activity_count"] > 5:
        return "large_engagement"
    elif row["unique_types"] >= 3:
        return "multi_activity"
    elif row["activity_count"] > 3:
        return "engaged"
    elif row["activity_count"] == 1:
        return "single_event"
    else:
        return "light_moment"

cluster_summary["moment_type"] = cluster_summary.apply(classify_moment, axis=1)
cluster_summary = cluster_summary[cluster_summary["cluster"] != -1]
df_all_with_clusters = df_all[df_all["cluster"] != -1]

cluster_summary.head(), df_all_with_clusters.head()


ValueError: invalid literal for int() with base 10: '2023-01-01T08:18:00'